### P2.3 — Deep Learning Foundations | Python to Gen AI Course
### P2.3.9 — CNN — Convolutional Neural Network

---
## 🔁 Quick Recap — Where We Are

In P2.3.8 we built an ANN for tabular data.  
ANN works beautifully when your features are structured rows and columns.

But what happens when your data is an **image?**

A 28×28 image has 784 pixels.  
If you flatten it and feed it to an ANN — you get 784 inputs.  
But you also **lose all the spatial information** — which pixel is next to which.

For a face recognition system, knowing that the eyes are above the nose  
is critical information. ANN throws that away.

**CNN was built specifically to keep that spatial structure.**

---
## 👁️ Why ANN Fails on Images

```
Original image (28×28):

  0  0  0  0  0  0  0
  0  0 255 255 0  0  0
  0 255  0  0 255 0  0
  0 255 255 255 255 0  0
  0 255  0  0  255 0  0
  0  0  0  0  0  0  0

ANN flattens this to:
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 255, 255, 0, 0, 0, ...]
  → 784 numbers in a row
  → The 2D structure is gone
  → ANN has no idea which pixels are neighbours
```

Additionally — a full ANN on images has a **parameter explosion:**
```
28×28 image → 784 inputs
First hidden layer of 128 neurons
→ 784 × 128 = 100,352 weights just for one layer

A 224×224 colour image → 150,528 inputs
→ Millions of weights in the first layer alone
→ Computationally impossible at scale
```

CNN solves both problems.

<img src="ann_vs_cnn_image.png" width="800"/>
<!-- Visual: LEFT — image being flattened into a 1D array going into an ANN. Red X over it labeled 'Loses spatial structure'. RIGHT — same image going directly into a CNN (shown as a 3D block) preserving its grid shape. Green checkmark labeled 'Preserves spatial structure'. -->

---
## 🔍 The Key Idea — Convolution (Filters)

Instead of connecting every pixel to every neuron —  
CNN uses small **filters** (also called kernels) that slide across the image.

```
A filter is a small grid — e.g. 3×3

  [ 1  0 -1 ]
  [ 1  0 -1 ]  ← This filter detects vertical edges
  [ 1  0 -1 ]
```

The filter slides across the image, one step at a time.  
At each position, it computes a dot product with the pixels underneath it.  
The result is a new grid called a **Feature Map** — showing where that pattern appears.

```
Image           Filter slides          Feature Map
(28×28)         across image     →     (detects edges)

  ┌─────────┐                          ┌─────────┐
  │  pixels │   [3×3 filter]           │  edges  │
  │  ██ ██  │ ──────────────────────▶  │ detected│
  │  ██████ │                          │         │
  └─────────┘                          └─────────┘
```

**One CNN layer uses many filters in parallel:**
```
Filter 1  →  detects vertical edges
Filter 2  →  detects horizontal edges
Filter 3  →  detects diagonal edges
Filter 4  →  detects curves
...
32 filters → 32 feature maps — one per pattern
```

The CNN **learns** what patterns each filter should detect —  
it is not hand-coded. The weights in each filter are learned during training.

<img src="convolution.png" width="800"/>
<!-- Visual: animated-style diagram. 5×5 image on the left. A 3×3 filter box sliding across it with position highlighted. Output feature map on the right building up as the filter slides. Number computed at each position shown (dot product). Arrow from image to feature map. -->

---
## 🌊 Pooling — Shrinking While Keeping What Matters

After convolution, we have feature maps — but they can still be large.  
**Pooling** reduces the size while keeping the most important information.

The most common type is **Max Pooling:**

```
Take a 2×2 window, slide across the feature map
At each position — keep only the maximum value

Before Max Pooling (4×4):        After Max Pooling (2×2):
  1   3   2   4                     3   4
  5   6   1   2       ──────▶        6   4
  3   2   4   1
  1   5   2   3
```

Why does this work?  
If an edge is detected somewhere in a 2×2 region — the exact pixel does not matter.  
The max value captures **whether** the feature is present, not exactly where.

**Benefits of Pooling:**
```
✅  Reduces the size of feature maps — fewer parameters downstream
✅  Reduces computation
✅  Makes the detection slightly position-invariant
   (a slightly shifted eye is still recognised as an eye)
```

<img src="max_pooling.png" width="700"/>
<!-- Visual: 4×4 grid on the left. 2×2 windows highlighted in sequence. Each window shows its maximum value being selected. Result 2×2 grid on the right. Arrow connecting them. Label: 'Max Pooling — keep the strongest signal in each region'. -->

---
## 🏗️ Full CNN Architecture

A typical CNN has two stages:

```
Stage 1 — Feature Extraction
  Conv Layer  →  ReLU  →  Pool
  Conv Layer  →  ReLU  →  Pool
  (repeat as needed)

Stage 2 — Classification (exactly like ANN)
  Flatten  →  Dense Layer  →  Dense Output
```

The first stage learns **what** is in the image (features).  
The second stage learns **how to classify** based on those features.

```
Input Image
    ↓
Conv(32 filters) + ReLU   ← learns 32 low-level patterns (edges)
    ↓
MaxPool
    ↓
Conv(64 filters) + ReLU   ← learns 64 higher-level patterns (shapes)
    ↓
MaxPool
    ↓
Flatten                   ← convert 2D feature maps to 1D
    ↓
Dense(128) + ReLU         ← classify based on detected features
    ↓
Dense(10) + Softmax       ← 10 output classes (e.g. digit 0-9)
```

<img src="cnn_architecture.png" width="850"/>
<!-- Visual: full CNN pipeline. Image on far left → Conv block (3D feature map) → Pool block (smaller 3D) → Conv block → Pool block → Flatten (1D vector) → Dense layers → Output probabilities. Each stage labeled. Show feature maps getting smaller but deeper (more filters). -->

---
## 💻 Let's Build It — CNN for Image Classification

We use the **MNIST dataset** — 70,000 images of handwritten digits (0–9).  
Each image: 28×28 pixels, grayscale.

Task: Multi-class classification — which digit is this?

```
Input  → 28×28 grayscale image
Output → 10 classes (digit 0 to 9)
```

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score

# ── DATA ──────────────────────────────────────────────────────────
# MNIST is built into Keras — 70,000 labeled digit images
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Train images : {X_train.shape}  → {X_train.shape[0]} images of {X_train.shape[1]}×{X_train.shape[2]} pixels")
print(f"Test  images : {X_test.shape}")
print(f"Labels       : {np.unique(y_train)}  (digits 0 to 9)")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1 : SPLIT + PREPARE
# MNIST is already split into train/test — we just prepare it
# ══════════════════════════════════════════════════════════════════

# Reshape: CNN expects (samples, height, width, channels)
# Grayscale = 1 channel  (colour images would be 3 channels — RGB)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Normalise pixel values from 0-255 to 0-1
# Same idea as StandardScaler — keeps all values on a similar scale
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print("Data prepared:")
print(f"  Shape after reshape : {X_train.shape}")
print(f"  Pixel values range  : {X_train.min()} to {X_train.max()}  (normalised)")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 2 : TRAIN
# ══════════════════════════════════════════════════════════════════

# ── BUILD CNN ─────────────────────────────────────────────────────
model = keras.Sequential([

    # Stage 1 — Feature Extraction
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    # 32 filters, each 3×3 → learns 32 different patterns
    layers.MaxPooling2D((2,2)),
    # Shrink feature maps by half

    layers.Conv2D(64, (3,3), activation='relu'),
    # 64 filters → learns higher-level patterns from the first layer's output
    layers.MaxPooling2D((2,2)),

    # Stage 2 — Classification
    layers.Flatten(),
    # Convert 2D feature maps to 1D — ready for Dense layers

    layers.Dense(128, activation='relu'),
    # Fully connected — learns how to classify from detected features

    layers.Dense(10, activation='softmax')
    # 10 output neurons — one per digit (0-9)
    # Softmax → probabilities that sum to 1
])

# ── COMPILE ───────────────────────────────────────────────────────
# sparse_categorical_crossentropy — multi-class classification
# Labels are integers (0-9) not one-hot — 'sparse' handles this
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── TRAIN ─────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,         # 5 epochs is enough for MNIST to reach ~99%
    batch_size=64,
    verbose=1
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 3 : EVALUATE
# ══════════════════════════════════════════════════════════════════
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss,  test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

train_acc = round(train_acc * 100, 2)
test_acc  = round(test_acc  * 100, 2)

print("=== PHASE 3 : EVALUATE ===")
print(f"  Train Accuracy : {train_acc}%")
print(f"  Test  Accuracy : {test_acc}%")

gap = train_acc - test_acc
if gap < 2:
    print("  ✅ Good Fit — train and test scores are close")
elif gap > 10:
    print("  ⚠️  Overfitting — model memorised training data")
else:
    print("  ✅ Acceptable — small gap, generalises well")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 4 : INFERENCE
# ══════════════════════════════════════════════════════════════════

# Predict on the first 5 test images
sample_images = X_test[:5]
sample_labels = y_test[:5]

predictions = model.predict(sample_images, verbose=0)
predicted_digits = np.argmax(predictions, axis=1)

print("=== PHASE 4 : INFERENCE ===")
print("Actual vs Predicted for first 5 test images:")
for i in range(5):
    confidence = round(predictions[i][predicted_digits[i]] * 100, 1)
    status = "✅" if predicted_digits[i] == sample_labels[i] else "❌"
    print(f"  Image {i+1}: Actual={sample_labels[i]}  Predicted={predicted_digits[i]}  Confidence={confidence}%  {status}")

---
## 🌍 Where CNN Is Used in the Real World

```
────────────────────────────────────────────────────────────────
 Application           What CNN Does
────────────────────────────────────────────────────────────────
 Face ID (iPhone)      Recognises your face from pixel patterns
 Google Photos         Identifies people, places, objects in photos
 Medical Imaging       Detects tumours in X-rays, MRI, retina scans
 Tesla Autopilot       Reads road lanes, signs, pedestrians in real time
 Instagram Filters     Detects face landmarks and applies effects
 Document Scanning     Reads and digitises text from photos (OCR)
 Satellite Imagery     Classifies land use, detects ships, crops, fires
────────────────────────────────────────────────────────────────
```

<img src="cnn_real_world.png" width="750"/>
<!-- Visual: 3×2 icon grid. Each tile: application icon + one-line label. Face ID / Google Photos / Medical Scan / Tesla / Instagram / Satellite. Dark background, clean tile layout. -->

---
## ✅ Summary — What You Learned

| Concept | Key Point |
|---|---|
| Why ANN fails on images | Flattening loses spatial structure — pixels lose their neighbours |
| Filter (Kernel) | Small grid that slides across the image detecting one pattern |
| Feature Map | Output of one filter — shows where a pattern appears |
| Max Pooling | Shrinks feature maps — keeps strongest signal per region |
| CNN Architecture | Conv+ReLU+Pool (feature extraction) → Flatten → Dense (classification) |
| MNIST | 70,000 handwritten digit images — classic CNN benchmark |
| Softmax output | 10 neurons for 10 classes — probabilities summing to 1 |
| Real world | Face recognition, medical imaging, autonomous driving |

---

**👉 Next: P2.3.10 — RNN and LSTM — networks with memory, built for sequences**